# eXpairing — Food.com Dataset EDA

This notebook explores the Food.com dataset to understand:
- Recipe and rating distributions
- Ingredient frequency and vocabulary
- Cold-start challenge (sparse users)
- β calibration: how many missing ingredients do users actually cook with?

Run this after `seed_recipes.py` and `seed_ratings.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter
from sqlalchemy import text

from backend.db.database import SessionLocal
from backend.db.models import Recipe, UserEvent, User

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi']     = 110

db = SessionLocal()
print("DB connected")

## 1. Dataset overview

In [ ]:
n_recipes = db.query(Recipe).count()
n_ratings = db.query(UserEvent).filter(UserEvent.event_type == 'rate').count()
n_users   = db.execute(text("SELECT COUNT(DISTINCT user_id) FROM user_events WHERE event_type='rate'")).scalar()

print(f"Recipes  : {n_recipes:,}")
print(f"Ratings  : {n_ratings:,}")
print(f"Users    : {n_users:,}")
print(f"Density  : {n_ratings / (n_recipes * n_users) * 100:.4f}%  (fraction of possible ratings filled)")

## 2. Rating distribution

Understanding the rating distribution tells us the bias in our training data.
Food.com ratings are heavily skewed toward 5 stars — this is called **positivity bias**.
The SVD model mean-centers per user to correct for this.

In [ ]:
rows = db.execute(text("SELECT rating, COUNT(*) as cnt FROM user_events WHERE event_type='rate' GROUP BY rating ORDER BY rating")).fetchall()
ratings_df = pd.DataFrame(rows, columns=['rating', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(ratings_df['rating'], ratings_df['count'], color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
axes[0].set_xlabel('Rating (stars)')
axes[0].set_ylabel('Number of ratings')
axes[0].set_title('Rating distribution')
for i, row in ratings_df.iterrows():
    axes[0].text(row['rating'], row['count'] + ratings_df['count'].max()*0.01,
                 f"{row['count']/ratings_df['count'].sum()*100:.1f}%", ha='center', fontsize=9)

# Cumulative
ratings_df['pct'] = ratings_df['count'] / ratings_df['count'].sum() * 100
ratings_df['cum'] = ratings_df['pct'].cumsum()
axes[1].plot(ratings_df['rating'], ratings_df['cum'], marker='o', color='steelblue')
axes[1].axhline(50, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Rating (stars)')
axes[1].set_ylabel('Cumulative %')
axes[1].set_title('Cumulative rating distribution')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean rating: {sum(r*c for r,c in zip(ratings_df.rating, ratings_df['count'])) / ratings_df['count'].sum():.2f}")

## 3. Ratings per user — cold-start analysis

This is the key slide for the CF section of the presentation.
It shows how many users have fewer than 5 ratings (cold-start territory)
vs how many have enough for SVD predictions.

In [ ]:
user_counts = pd.read_sql(
    "SELECT user_id, COUNT(*) as n_ratings FROM user_events WHERE event_type='rate' GROUP BY user_id",
    db.get_bind()
)

MIN_RATINGS_FOR_CF = 5

cold_users = (user_counts['n_ratings'] < MIN_RATINGS_FOR_CF).sum()
warm_users = (user_counts['n_ratings'] >= MIN_RATINGS_FOR_CF).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(user_counts['n_ratings'].clip(upper=50), bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(MIN_RATINGS_FOR_CF, color='red', linestyle='--', linewidth=1.5,
                label=f'CF threshold ({MIN_RATINGS_FOR_CF} ratings)')
axes[0].set_xlabel('Ratings per user (capped at 50)')
axes[0].set_ylabel('Number of users')
axes[0].set_title('Ratings per user')
axes[0].legend()

# Cold/warm pie
axes[1].pie([cold_users, warm_users],
            labels=[f'Cold start\n({cold_users:,} users)', f'Warm CF\n({warm_users:,} users)'],
            colors=['#3498db', '#2ecc71'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Cold start vs warm users\n(threshold = {MIN_RATINGS_FOR_CF} ratings)')

plt.tight_layout()
plt.savefig('cold_start_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Cold start users (<{MIN_RATINGS_FOR_CF} ratings): {cold_users:,} ({cold_users/len(user_counts)*100:.1f}%)")
print(f"Warm CF users   (≥{MIN_RATINGS_FOR_CF} ratings): {warm_users:,} ({warm_users/len(user_counts)*100:.1f}%)")
print(f"Median ratings per user: {user_counts['n_ratings'].median():.0f}")

## 4. Ingredient frequency

The most common ingredients define our TF-IDF vocabulary.
This also shows what the pantry-based recommender will most often match on.

In [ ]:
rows = db.execute(text("SELECT ingredients_csv FROM recipes LIMIT 50000")).fetchall()
all_ingredients = []
for (csv,) in rows:
    all_ingredients.extend(i.strip() for i in csv.split(',') if i.strip())

counter = Counter(all_ingredients)
top_40 = counter.most_common(40)
labels = [i for i, _ in top_40]
counts = [c for _, c in top_40]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels[::-1], counts[::-1], color='#3a9e68')
ax.set_xlabel('Recipes containing ingredient')
ax.set_title('Top 40 most common ingredients (Food.com)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + counts[0]*0.005, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('ingredient_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Unique ingredients: {len(counter):,}")
print(f"Top ingredient '{top_40[0][0]}' appears in {top_40[0][1]:,} recipes")

## 5. Recipe cooking time distribution

Helps calibrate the 'quick' tag filter on the profile page.

In [ ]:
minutes = pd.read_sql(
    "SELECT minutes FROM recipes WHERE minutes IS NOT NULL AND minutes > 0 AND minutes < 300",
    db.get_bind()
)['minutes']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(minutes, bins=60, color='#e67e22', edgecolor='white')
ax.axvline(30, color='steelblue', linestyle='--', linewidth=1.5, label='30 min')
ax.axvline(60, color='red',       linestyle='--', linewidth=1.5, label='60 min')
ax.set_xlabel('Cooking time (minutes)')
ax.set_ylabel('Number of recipes')
ax.set_title('Cooking time distribution (< 300 min)')
ax.legend()
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x)}m'))

plt.tight_layout()
plt.savefig('cooking_times.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median cooking time: {minutes.median():.0f} min")
print(f"<30 min recipes: {(minutes <= 30).sum():,} ({(minutes <= 30).mean()*100:.1f}%)")
print(f"<60 min recipes: {(minutes <= 60).sum():,} ({(minutes <= 60).mean()*100:.1f}%)")

## 6. Tag distribution

Shows what dietary/category tags are available and how well represented each is.

In [ ]:
tag_rows = db.execute(text("SELECT tags_csv FROM recipes WHERE tags_csv IS NOT NULL LIMIT 100000")).fetchall()
all_tags = []
for (csv,) in tag_rows:
    all_tags.extend(t.strip() for t in csv.split(',') if t.strip())

tag_counter = Counter(all_tags)
diet_tags = ['vegetarian','vegan','gluten-free','low-calorie','low-fat',
             'healthy','low-sodium','low-cholesterol','low-carb','diabetic']

diet_counts = {t: tag_counter.get(t, 0) for t in diet_tags}

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ecc71' if c > 1000 else '#f39c12' if c > 100 else '#e74c3c'
          for c in diet_counts.values()]
bars = ax.bar(list(diet_counts.keys()), list(diet_counts.values()), color=colors)
ax.set_xlabel('Diet tag')
ax.set_ylabel('Recipes with this tag')
ax.set_title('Dietary tag coverage in Food.com')
ax.tick_params(axis='x', rotation=35)
for bar, val in zip(bars, diet_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('tag_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. β calibration — how many missing ingredients do users actually cook with?

This informs the default β value and the cold-start seed threshold.
If most users cook with 0-1 missing ingredients, a high default β is appropriate.

In [ ]:
# Using our dev seed events as a proxy (real data after seeding)
events = pd.read_sql(
    "SELECT n_missing FROM user_events WHERE event_type='cook' AND n_missing IS NOT NULL",
    db.get_bind()
)

if len(events) == 0:
    print("No cook events yet — add some by cooking recipes in the app, then re-run.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    events['n_missing'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#9b59b6')
    ax.set_xlabel('Missing ingredients when cooking')
    ax.set_ylabel('Number of cook events')
    ax.set_title('How many missing ingredients do users actually cook with?')
    ax.tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig('missing_ingredients_behavior.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Mean missing: {events['n_missing'].mean():.2f}")
    print(f"Median missing: {events['n_missing'].median():.0f}")
    print(f"% cooked with 0 missing: {(events['n_missing'] == 0).mean()*100:.1f}%")

## Summary — key numbers for the presentation

Run this cell last to get a clean summary table.

In [ ]:
from datetime import datetime

n_recipes   = db.query(Recipe).count()
n_ratings   = db.execute(text("SELECT COUNT(*) FROM user_events WHERE event_type='rate'")).scalar()
n_users     = db.execute(text("SELECT COUNT(DISTINCT user_id) FROM user_events WHERE event_type='rate'")).scalar()
avg_rating  = db.execute(text("SELECT AVG(rating) FROM user_events WHERE event_type='rate'")).scalar()
n_tags      = db.execute(text("SELECT COUNT(DISTINCT value) FROM (SELECT TRIM(value) as value FROM recipes, json_each('['||REPLACE(tags_csv,',','\",')||'\"' ||']'))")).scalar() if False else '—'

print("=" * 45)
print("  eXpairing — Dataset Summary")
print("=" * 45)
print(f"  Recipes          : {n_recipes:>10,}")
print(f"  Ratings          : {n_ratings:>10,}")
print(f"  Users (rated)    : {n_users:>10,}")
print(f"  Avg rating       : {(avg_rating or 0):>10.2f} / 5.0")
print(f"  Density          : {n_ratings/(n_recipes*max(n_users,1))*100:>9.4f}%")
print("=" * 45)
print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

db.close()